# Нормализация данных перед анализом

#### Выполнил X

Подведем краткий итог предшествовавших этапов, посвященных поиску / подготовке данных:

<ol>
    <li>
        В датасете содержалось много дубликатов ≈ 500 строк — они были удалены. Пропуски по рейтингу фильма были заполнены медианой по возрастным группам. Названия колонок были записаны в формате <i>snake_case</i>.
    </li>
    <li>
        Были подключены сторонние датасеты с большим числом строк. Затем последовало обогащение начального датасета новыми признаками: budget, genre, revenue, runtime, vote_average_imdb, vote_average_tmdb, vote_count_tmdb, vote_count_imdb и производными от них признаками в рамкам feature engeneering. Некоторые колонки начального датасета были переименованы. Преимуществом данного этапа является высокая уверенность в данных, поскольку датасеты объединялись по точному совпадению названия проекта и года выхода. 
    </li>
    <li>
        После предыдущего этапа осталось большое число пропусков, поскольку данных о некоторых проектах в сторонних датасетах не содержалось / они были неполными / их тяжело получить по названию + году. Наша команда решила обратиться к парсингу данных со сторонних агрегаторов: IMDB, Box Office Mojo и The Numbers. Недостающие данные были собраны, однако их точность на этот раз не абсолютна. Тем не менее наша команда принимает это допущение в рамках анализа, поскольку оно не мешает выполнить поставленную задачу — предоставить ответ на вопрос: «Какой фильм / сериал стоит снять студии».
    </li>
</ol>

Перейдем к рассмотрению упомянутых допущений.

<ul>
    <li>Во время парсинга наша команда полагалась на внутреннюю поисковую систему агрегаторов (особенно явно это прослеживается в случае с IMDB). Внутренний поиск агрегаторов оказался удобен и точен, однако он показывал самые популярные результаты. Например, в случае с сериалом он показывал данные за все сезоны, хотя в начальном датасете содержалась ссылка на конкретный сезон / некоторые данные найти было невозможно. С другой стороны, самая актуальная информация зачастую как раз и представлена на подобных агреггированных страницах. Зрители склонны редко оценивать все сезоны сериалов. Например, в случае с «Prison Break» на странице с 3 сезоном всего 1000 отзывов, тогда как на титульной странице около 700 тыс. отзывов. Наша команда считает, что именно вторая цифра отражает реальную популярность / ценность медиапродукта и является более полезной в рамках поставленной задачи.</li>
    <li>
        Все собранные данные современны и могут не отражать действительность в момент появления начального датасета. Например, знаменитый сериал «Breaking Bad» закончил сниматься в начале 2010-х годов, а данные по нему были собраны  нашей командой в 2026 году — оценка могла повыситься / понизиться, число оценивших сериал выросло. Тем не менее это допущение не мешает выполнить поставленную задачу, поскольку в рамках анализа для нашей команды имеют большую ценность не абсолютные, а относительные показатели: лучше / хуже, больше / меньше. Проект, вышедший 10 лет назад и не запомнившийся зрителям, получит больше голосов, его оценка может измениться, но это произойдет и с остальными проектами — итоговое распределение вряд ли сильно исказиться: популярные и значимые медиапродукты останутся таковыми, аутсайдеры внезапно не завоюют популярность.
    </li>
</ul>

В рамках данного этапа будет проведена нормализация собранных данных перед анализом и приведение их к единому виду

## Импорт библиотек

In [34]:
import pandas as pd
from typing import Any, List
import numpy as np

## Подключение датасетов

In [35]:
df_result = pd.read_csv('parsed_imdb_2.csv')
df_imdb_parsed = pd.read_csv('parsed_imdb_1.csv')
df_mojo_parsed = pd.read_csv('parsed_mojo.csv')

## Анализ результатов

Во время парсинга наша команда пыталась найти данные для следуюших признаков:

<ul>
    <li><code>budget</code></li>
    <li><code>revenue</code></li>
    <li><code>runtime</code></li>
    <li><code>genres</code></li>
    <li><code>vote_count_...</code> (число голосов, любой агреггатор)</li>
    <li><code>rating_...</code> (оценка зрителей, любой агреггатор)</li>
<ul>

В данный момент у нас есть датасета, который структурно повторяет начальный: `df_result`.

## Первичная обработка базового датасета

Обработаем базовый датасет перед объединением. Во-первых, удалим лишние признаки:

<ul>
    <li><code>rating_level</code> — описание возрастного рейтинга, не нужно для анализа
    </li>
    <li><code>rating_size_nfx</code> — менее ценный признак, чем <code>imdb_votecount_new</code></li>
    <li><code>title_lower</code> — был нужен только для обогащение датасета на втором шаге</li>
    <li><code>imdb_id</code> — был нужен только для парсинга</li>
    <li><code>vote_average_tmdb</code> — было решено использовать оценку IMDB, поскольку по ней меньше пропусков</li>
    <li><code>vote_count_tmdb</code> — было решено использовать число голосов IMDB, поскольку по ним меньше пропусков</li>
    <li><code>vote_count_imdb</code> — признак <code>imdb_votecount_new</code> актуальнее</li>
</ul>

In [36]:
df_base = df_result.drop(columns=
    [
        'rating_level',
        'rating_size_nfx',
        'title_lower',
        'imdb_id',
        'vote_average_tmdb',
        'vote_count_tmdb',
        'vote_count_imdb'
    ]
)

df_base = df_base.iloc[:, 1:]

df_base.head(5)

,title,rating,release_year,rating_score_nfx,budget,revenue,runtime,vote_average_imdb,genres,genres_list,genres_consolidated,is_action,is_comedy,is_drama,is_family,is_other,imdb_votecount_new
0,White Chicks,PG-13,2004,82.0,37000000.0,113086475.0,109.0,5.9,"Comedy, Crime","['Comedy', 'Crime']","['Action', 'Comedy']",1,1,0,0,0,193.7K
1,Lucky Number Slevin,R,2006,79.0,27000000.0,56308881.0,110.0,7.7,"Drama, Thriller, Crime, Mystery","['Drama', 'Thriller', 'Crime', 'Mystery']","['Action', 'Drama', 'Other']",1,0,1,0,1,339K
2,Death Note,TV-14,2006,77.0,20000000.0,29667169.0,126.0,7.5,"Fantasy, Mystery, Thriller","['Fantasy', 'Mystery', 'Thriller']","['Action', 'Other']",1,0,0,0,1,481.2K
3,The Hunter,R,2011,79.0,NaN,176669.0,102.0,6.7,"Drama, Thriller, Adventure","['Drama', 'Thriller', 'Adventure']","['Action', 'Drama']",1,0,1,0,0,82.1K
4,The Do-Over,TV-MA,2016,84.0,NaN,NaN,108.0,5.7,"Action, Adventure, Comedy","['Action', 'Adventure', 'Comedy']","['Action', 'Comedy']",1,1,0,0,0,57.1K


## Объединение датасетов

In [37]:
df_imdb_parsed = df_imdb_parsed[
    [
        'title',
        'parsed_budget_imdb', 
        'parsed_genres_imdb',
        'parsed_runtime_imdb',
        'parsed_vote_count_imdb'
    ]
]


df_imdb_parsed = df_imdb_parsed.rename(
    columns={
        'parsed_budget_imdb': 'budget',
        'parsed_genres_imdb': 'genres',
        'parsed_runtime_imdb': 'runtime',
        'parsed_vote_count_imdb': 'imdb_votecount_new'
    }
)

df_base = pd.merge(
    left=df_base,
    right=df_imdb_parsed,
    how='left',
    on='title',
    suffixes=('', '_parsed')
)

new_columns = ['budget', 'genres', 'runtime', 'imdb_votecount_new']

for column in new_columns:
    df_base[column] = df_base[column].fillna(df_base[f"{column}_parsed"])

df_base = df_base.drop(columns=list(map(lambda x: f"{x}_parsed", new_columns)))

In [38]:
df_mojo_parsed = df_mojo_parsed[
    [
        'title',
        'parsed_revenue_mojo',
        'parsed_budget_mojo'
    ]
]


df_mojo_parsed = df_mojo_parsed.rename(
    columns={
        'parsed_revenue_mojo' : 'revenue',
        'parsed_budget_mojo' : 'budget'
    }
)

df_base = pd.merge(
    left=df_base,
    right=df_mojo_parsed,
    how='left',
    on='title',
    suffixes=('', '_parsed')
)

new_columns = ['budget', 'revenue']

for column in new_columns:
    df_base[column] = df_base[column].fillna(df_base[f"{column}_parsed"])

df_base = df_base.drop(columns=list(map(lambda x: f"{x}_parsed", new_columns)))

df_base.isna().sum()

title                    0
rating                   0
release_year             0
rating_score_nfx         0
budget                 370
revenue                362
runtime                 18
vote_average_imdb       18
genres                  25
genres_list              0
genres_consolidated      0
is_action                0
is_comedy                0
is_drama                 0
is_family                0
is_other                 0
imdb_votecount_new      12
dtype: int64

## Нормализация признаков

Нормализуем признак `rating`:

In [39]:
df_base['rating'].unique()

array(['PG-13', 'R', 'TV-14', 'TV-MA', 'NR', 'PG', 'TV-G', 'G', 'TV-PG',
       'TV-Y7-FV', 'TV-Y', 'TV-Y7'], dtype=object)

После изучения каждого значения было решено разбить начальные 12 значений на 3 категории: babies / teens / adults — дети / подростки / взрослые

In [40]:
rating_map = {
    'G': 'kids',
    'TV-G': 'kids',
    'TV-Y': 'kids',
    'TV-Y7': 'kids',
    'TV-Y7-FV': 'kids',

    'PG': 'teens',
    'TV-PG': 'teens',
    'PG-13': 'teens',
    'TV-14': 'teens',

    'R': 'adults',
    'TV-MA': 'adults',
    'NR': 'adults'
}

df_base['rating'] = df_base['rating'].apply(lambda x: rating_map[x])
df_base['rating'].value_counts()

rating
teens     230
kids      169
adults    105
Name: count, dtype: int64

Замена прошла успешно. По каждому классу есть данные

Нормализуем признак `imdb_votecount_new`:

In [41]:
def normalize_votecount(x: Any) -> float | int:
    if pd.isna(x): return x

    try:
        return int(x)
    except (ValueError, TypeError):
        s: str = str(x)
        s = s.replace('K', '')
        value = float(s) * 1000
        return int(value)

In [42]:
df_base['imdb_votecount_new'] = df_base['imdb_votecount_new'].apply(lambda x: normalize_votecount(x))

In [43]:
df_base.head(5)

,title,rating,release_year,rating_score_nfx,budget,revenue,runtime,vote_average_imdb,genres,genres_list,genres_consolidated,is_action,is_comedy,is_drama,is_family,is_other,imdb_votecount_new
0,White Chicks,teens,2004,82.0,37000000.0,113086475.0,109.0,5.9,"Comedy, Crime","['Comedy', 'Crime']","['Action', 'Comedy']",1,1,0,0,0,193700.0
1,Lucky Number Slevin,adults,2006,79.0,27000000.0,56308881.0,110.0,7.7,"Drama, Thriller, Crime, Mystery","['Drama', 'Thriller', 'Crime', 'Mystery']","['Action', 'Drama', 'Other']",1,0,1,0,1,339000.0
2,Death Note,teens,2006,77.0,20000000.0,29667169.0,126.0,7.5,"Fantasy, Mystery, Thriller","['Fantasy', 'Mystery', 'Thriller']","['Action', 'Other']",1,0,0,0,1,481200.0
3,The Hunter,adults,2011,79.0,NaN,176669.0,102.0,6.7,"Drama, Thriller, Adventure","['Drama', 'Thriller', 'Adventure']","['Action', 'Drama']",1,0,1,0,0,82100.0
4,The Do-Over,adults,2016,84.0,NaN,NaN,108.0,5.7,"Action, Adventure, Comedy","['Action', 'Adventure', 'Comedy']","['Action', 'Comedy']",1,1,0,0,0,57100.0


Нормализуем `genres`. Будем придерживаться принятого разбиения на жанры.

In [44]:
df_base = df_base.drop(
    columns=[
        'genres_list', 
        'genres_consolidated',
        'is_action',
        'is_comedy',
        'is_drama',
        'is_family',
        'is_other'
    ]
)

def normalize_genres(x: str) -> List[str] | float:
    if pd.isna(x): return np.nan

    genres_list = x.split(', ')
    return genres_list if len(genres_list) < 2 else genres_list[:2]

# Оставляем только главных два жанра
df_base['genres'] = df_base['genres'].apply(normalize_genres)
df_base.head(5)

,title,rating,release_year,rating_score_nfx,budget,revenue,runtime,vote_average_imdb,genres,imdb_votecount_new
0,White Chicks,teens,2004,82.0,37000000.0,113086475.0,109.0,5.9,"[Comedy, Crime]",193700.0
1,Lucky Number Slevin,adults,2006,79.0,27000000.0,56308881.0,110.0,7.7,"[Drama, Thriller]",339000.0
2,Death Note,teens,2006,77.0,20000000.0,29667169.0,126.0,7.5,"[Fantasy, Mystery]",481200.0
3,The Hunter,adults,2011,79.0,NaN,176669.0,102.0,6.7,"[Drama, Thriller]",82100.0
4,The Do-Over,adults,2016,84.0,NaN,NaN,108.0,5.7,"[Action, Adventure]",57100.0


Посмотрим на распределение данных внутря колонки `genres`:

In [45]:
df_base['genres'].explode().value_counts()

genres
Comedy             159
Animation          144
Drama              135
Adventure          108
Family              78
Action              72
Crime               44
Romance             33
Fantasy             23
Horror              15
Documentary         13
Thriller            13
Mystery             11
Short                7
Biography            4
Music                3
History              3
War                  2
Science Fiction      2
Reality-TV           2
Talk-Show            2
Adult                2
TV Movie             1
News                 1
Game-Show            1
Sport                1
Musical              1
Name: count, dtype: int64

Оставим 6 категорий по смыслу:

In [46]:
family = [
    'Animation',
    'Family'
]

action = [
    'Action',
    'Adventure',
    'Science Fiction',
    'War',
    'Sport'
]

drama = [
    'Drama',
    'Romance',
    'Biography',
    'History',
    'Music',
    'Musical'
]

comedy = [
    'Comedy'
]

thriller = [
    'Crime',
    'Thriller',
    'Mystery',
    'Horror',
    'Fantasy'
]

other = [
    'Documentary',
    'Short',
    'Reality-TV',
    'News',
    'Game-Show',
    'TV Movie',
    'Adult',
    'Talk-Show'
]

genres_mapper = {
    "is_family": family,
    'is_action': action,
    "is_drama": drama,
    "is_comedy": comedy,
    "is_thriller": thriller,
    "is_other": other
}

In [47]:
new_genres_columns = list(genres_mapper.keys())

def fill_genres(x: pd.Series, column: str) -> bool | float:
    
    if not isinstance(x['genres'], List):
        return np.nan
    
    genres = set(x['genres'])
    mapper = set(genres_mapper[column])

    return bool(mapper.intersection(genres))

for column in new_genres_columns:
    df_base[column] = df_base.apply(fill_genres, axis=1, args=(column,))

In [48]:
df_base.head(5)

,title,rating,release_year,rating_score_nfx,budget,revenue,runtime,vote_average_imdb,genres,imdb_votecount_new,is_family,is_action,is_drama,is_comedy,is_thriller,is_other
0,White Chicks,teens,2004,82.0,37000000.0,113086475.0,109.0,5.9,"[Comedy, Crime]",193700.0,False,False,False,True,True,False
1,Lucky Number Slevin,adults,2006,79.0,27000000.0,56308881.0,110.0,7.7,"[Drama, Thriller]",339000.0,False,False,True,False,True,False
2,Death Note,teens,2006,77.0,20000000.0,29667169.0,126.0,7.5,"[Fantasy, Mystery]",481200.0,False,False,False,False,True,False
3,The Hunter,adults,2011,79.0,NaN,176669.0,102.0,6.7,"[Drama, Thriller]",82100.0,False,False,True,False,True,False
4,The Do-Over,adults,2016,84.0,NaN,NaN,108.0,5.7,"[Action, Adventure]",57100.0,False,True,False,False,False,False


Нормализуем `runtime`:

In [49]:
def normilize_runtime(x: Any) -> float:
    if pd.isna(x): return np.nan

    try:
        value = float(x)
        return value
    
    except (ValueError, TypeError):
        runtime = x.strip()
        runtime_list = runtime.split(" ")
        runtime_result = 0

        for part in runtime_list:
            if "h" in part:
                runtime_result += int(part.replace("h", "")) * 60
            elif "m" in part:
                runtime_result += int(part.replace("m", ""))

        return float(runtime_result)
        

df_base['runtime'] = df_base['runtime'].apply(lambda x: normilize_runtime(x))

## Удаление последних пропусков

Удалим оставшиеся пропуски:

In [50]:
df_base.isna().sum()

title                   0
rating                  0
release_year            0
rating_score_nfx        0
budget                370
revenue               362
runtime                18
vote_average_imdb      18
genres                 25
imdb_votecount_new     12
is_family              25
is_action              25
is_drama               25
is_comedy              25
is_thriller            25
is_other               25
dtype: int64

Заменим пропуски в признаке `imdb_votecount_new` на медиану по возрастным группам:

In [51]:
df_base["vote_average_imdb"] = df_base["vote_average_imdb"].fillna(
    df_base.groupby("rating")["vote_average_imdb"].transform("median")
)

Заменим пропуски `runtime` на медиану и удалим ошибочные выбросы:

In [52]:
df_base['runtime'][df_base['runtime'] > 180] = np.nan
df_base['runtime'] = df_base['runtime'].fillna(df_base['runtime'].median())

C:\Users\Егор\AppData\Local\Temp\ipykernel_27320\3108746283.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_base['runtime'][df_base['runtime'] > 180] = np.nan


Пропуски по `genres` заполнить нельзя — для этого не хватает данных. Было принято решение избавиться от проблемных строк:

In [53]:
df_base = df_base.dropna(subset=["genres"])

Было принято решение оставить пропуски по полям `budget` и `revenue` для проведения ограниченного анализа

In [54]:
print(df_base.shape)
df_base.isna().sum()

(479, 16)


title                   0
rating                  0
release_year            0
rating_score_nfx        0
budget                345
revenue               337
runtime                 0
vote_average_imdb       0
genres                  0
imdb_votecount_new      0
is_family               0
is_action               0
is_drama                0
is_comedy               0
is_thriller             0
is_other                0
dtype: int64

## Проверка выбросов

In [55]:
def check_outliers_std(df: pd.DataFrame, column: str) -> str:

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[column] < lower) | (df[column] > upper)]

    return f'Число выбросов {column} — {outliers.shape[0]}'

In [56]:
check_outliers_std(df_base, 'runtime')

'Число выбросов runtime — 0'

In [57]:
check_outliers_std(df_base, 'imdb_votecount_new')


'Число выбросов imdb_votecount_new — 50'

In [58]:
check_outliers_std(df_base, 'budget')

'Число выбросов budget — 8'

In [59]:
check_outliers_std(df_base, 'revenue')

'Число выбросов revenue — 10'

Ликвидация выбросов по полю `imdb_votecount_new`. Устраним выбросы методом логарифмирования:

In [60]:
df_base["imdb_votecount_new"] = np.log1p(df_base["imdb_votecount_new"])
check_outliers_std(df_base, 'imdb_votecount_new')

'Число выбросов imdb_votecount_new — 0'

Было принято решение оставить пропуски по `budget` и `revenue`, поскольку эти признаки не используются в основном анализе

## Завершение этапа

Переименуем признак `imdb_votecount_new` в `imdb_votecount_log`, `rating` в `age_group`

In [61]:
df_base = df_base.rename(columns={
        'imdb_votecount_new': 'imdb_votecount_log',
        'rating': 'age_group'
    }
)

Удалим лишний признак `genres`:

In [62]:
df_base = df_base.drop(columns=['genres'])

In [63]:
df_base.head(5)

,title,age_group,release_year,rating_score_nfx,budget,revenue,runtime,vote_average_imdb,imdb_votecount_log,is_family,is_action,is_drama,is_comedy,is_thriller,is_other
0,White Chicks,teens,2004,82.0,37000000.0,113086475.0,109.0,5.9,12.174071,False,False,False,True,True,False
1,Lucky Number Slevin,adults,2006,79.0,27000000.0,56308881.0,110.0,7.7,12.733758,False,False,True,False,True,False
2,Death Note,teens,2006,77.0,20000000.0,29667169.0,126.0,7.5,13.084040,False,False,False,False,True,False
3,The Hunter,adults,2011,79.0,NaN,176669.0,102.0,6.7,11.315705,False,False,True,False,True,False
4,The Do-Over,adults,2016,84.0,NaN,NaN,108.0,5.7,10.952577,False,True,False,False,False,False


In [64]:
df_base.isna().sum()

title                   0
age_group               0
release_year            0
rating_score_nfx        0
budget                345
revenue               337
runtime                 0
vote_average_imdb       0
imdb_votecount_log      0
is_family               0
is_action               0
is_drama                0
is_comedy               0
is_thriller             0
is_other                0
dtype: int64

In [65]:
df_base.shape

(479, 15)

In [33]:
df_base.to_csv('netflix_normalized_3.csv')